# sim_multistage_field : Simulate multi-stage breeding trial data in inbred crops 
`sim_multistage_field( batch_years, n_crosses=100, n_lines=5000, n_env=10,  n_rep=3,var_components={'G': 0.30, 'GE': 0.25, 'e': 0.45}, rho_within=(0.5, 0.9),var_between_prop=0.4)`

Here is a detailed breakdown of every parameter in the updated `simulate_multistage_yield` function, grouped by how they control the trial design versus the underlying quantitative genetics.

### Trial Design Parameters

These parameters define the physical and temporal structure of your simulated breeding program.

* **`batch_years`** (Dictionary): Maps the name of a breeding batch to the specific sequence of years it is tested. This handles the staggered nature of the trial (e.g., `{'2014': [2018, 2019, 2020]}`).
* **`n_crosses`** (Integer): The number of distinct bi-parental families (crosses) generated in a single batch. Default is `100`.
* **`n_lines`** (Integer): The total number of individual progeny lines tested across the entire batch. The function automatically calculates the number of lines per cross by dividing `n_lines` by `n_crosses` (e.g., 5000 / 100 = 50 lines per family). Default is `5000`.
* **`n_env`** (Integer): The number of distinct geographic locations or testing environments utilized in a single year. Default is `10`.
* **`n_rep`** (Integer): The number of replications (blocks) for each line within a specific year-location combination. Default is `3`.

---

### Total Variance Components

This dictionary defines the absolute, true variation in the simulated population. These values represent the base standard deviations before environmental noise or specific cross architectures are applied.

* **`var_components`** (Dictionary): 
    * **`'G'` ($\sigma^2_G$)**: The total additive genetic variance in the population. This represents the total genetic diversity available for selection. Default is `0.30`.
    * **`'GE'` ($\sigma^2_{GE}$)**: The genotype-by-environment interaction variance. This dictates how much a line's performance fluctuates randomly across different locations and years. Default is `0.25`.
    * **`'e'` ($\sigma^2_e$)**: The residual error variance. This captures unexplained micro-environmental noise (e.g., field variation within a block) and measurement error. Default is `0.45`.

---

### Genetic Architecture Parameters

These are the most critical parameters for defining *how* the genetic variance ($\sigma^2_G$) is partitioned into family means versus Mendelian segregation.

* **`var_between_prop`** (Float: 0.0 to 1.0): 
    * **Definition:** The proportion of the total genetic variance ($\sigma^2_G$) that is assigned to the differences *between* the crosses (the family means).
    * **Mechanism:** If set to `0.4`, it means 40% of the total genetic variance dictates how different the 100 crosses are from one another. The remaining 60% of the genetic variance is reserved for the segregation of offspring within those crosses.
    * **Breeding Context:** A high value implies that parent selection is the absolute most critical step, as crosses are highly divergent.

* **`rho_within`** (Tuple: min, max): 
    * **Definition:** The range of the intra-family genetic correlation ($\rho$). For every cross, the simulation randomly assigns a $\rho$ value from within this range.
    * **Mechanism:** It controls the within-family variance using the formula: $\sigma^2_{within} = \sigma^2_G \times (1 - \rho)$. 
    * **Breeding Context:** * If $\rho$ is high (e.g., `0.9`), the siblings within that family are highly genetically correlated. The within-family variance is tiny, meaning the offspring will all cluster tightly around their specific family mean. 
        * If $\rho$ is low (e.g., `0.2`), the siblings are highly variable. You might have a family with a mediocre mean, but due to massive segregation, it could still contain a top-tier transgressive segregant.

In [ ]:
# Enable autoreload to sync changes made in the core files
%load_ext autoreload
%autoreload 2
# Import your simulation script directly
from python_core import sim_multistage_field

# Execute a function
trial_params = {
    '2014': [2018, 2019, 2020],
    '2015': [2019, 2020, 2021],
    '2016': [2020, 2021, 2022]
}
df_all = sim_multistage_field.sim_multistage_field(trial_params)

# Subsetting, Environment Creation, Mixed Model Fitting
subset_conditions = (
    ((df_all['Year'] == 2022) & df_all['Batch'].isin(['2014', '2015', '2016'])) |
    ((df_all['Year'] == 2021) & df_all['Batch'].isin(['2014', '2015'])) |
    ((df_all['Year'] == 2020) & df_all['Batch'].isin(['2013']))
)

df_subset = df_all[subset_conditions].copy()
print(df_subset)